# VAE Math Foundations

This companion notebook explains the math ideas behind `notebooks/05_vae.ipynb`. It is not a second VAE training notebook. The goal is to understand why the VAE code uses reconstruction loss, KL loss, `mu`, `logvar`, and the reparameterization trick.

## Learning purpose

Learn how VAE probability ideas make a latent space sampleable and how those ideas become the loss terms used in PyTorch code.

## Reference path

This notebook is designed to sit beside the practical notebook:

```text
notebooks/05_vae.ipynb
```

We will move slowly from ordinary autoencoders to the VAE loss function, then map the math back to the existing `loss_function()` implementation.


## 1. Ordinary autoencoder math

A regular autoencoder has two learned parts: an **encoder** and a **decoder**. The encoder compresses an input image into a smaller hidden representation. The decoder tries to rebuild the original image from that hidden representation.

```text
image x → encoder f(x) → latent code z → decoder g(z) → reconstruction x_hat
```

Simple formula view:

$$
z = f_\phi(x)
$$

$$
\hat{x} = g_\theta(z)
$$

Read this as: the encoder function $f_\phi$ turns the image $x$ into a latent code $z$, and the decoder function $g_\theta$ turns $z$ back into a reconstruction $\hat{x}$. The symbols $\phi$ and $\theta$ just mean the learned weights of the encoder and decoder.

Why use $\phi$ and $\theta$? They remind us that the functions are learned, not fixed. In this notebook, $\phi$ means the encoder parameters, which correspond to layers such as `fc1`, `fc21`, and `fc22`. The symbol $\theta$ means the decoder parameters, which correspond to layers such as `fc3` and `fc4`. Writing $f(x)$ and $g(z)$ is fine for intuition; writing $f_\phi(x)$ and $g_\theta(z)$ makes the trainable weights visible in the notation.

Meaning of the symbols:

- `x` is the original input image, such as one MNIST digit.
- `f(x)` is the encoder's output.
- `z` is the **latent code**, a smaller learned summary of the image.
- `g(z)` is the decoder's output.
- `x_hat` or $\hat{x}$ is the reconstructed version of the original image.

The important constraint is the **bottleneck**. The bottleneck is the smaller middle space where the model has fewer numbers than the original image. For MNIST, an input image has `28 × 28 = 784` pixel values. If the latent code has only 20 numbers, the model cannot simply store every pixel.

That pressure is useful. To reconstruct the image from fewer numbers, the model has to learn patterns that help rebuild many similar images, such as digit shape, stroke angle, loop size, and thickness. This is why autoencoders can be useful for compression, denoising, anomaly detection, and feature extraction.


### Reconstruction objective

A basic autoencoder trains by making the reconstruction `x_hat` close to the original input `x`.

```text
reconstruction loss = difference between x and x_hat
```

Simple formula view:

$$
\mathcal{L}_{\text{recon}} = d(x, \hat{x})
$$

Read this as: reconstruction loss is some distance or penalty $d$ between the original image $x$ and the rebuilt image $\hat{x}$. If the rebuilt image looks like the input image, the reconstruction loss is small. If the rebuilt image misses important pixels or changes the digit identity, the reconstruction loss is large.

This is the first half of the VAE story. A VAE still cares about reconstruction quality, but it changes the middle of the model from one fixed code into a small probability region.

```text
regular autoencoder:
x → one fixed latent code z → x_hat

variational autoencoder:
x → latent distribution q(z|x) → sampled latent code z → x_hat
```

Formula view of the VAE change:

$$
q_\phi(z \mid x)
$$

Read $q_\phi(z \mid x)$ as: the encoder's distribution of likely latent codes $z$, given this input image $x$.

A good plain-language explanation of the bottleneck is:

> The bottleneck forces the encoder to store only the most useful information in a smaller latent code, because the decoder must reconstruct the original image from that limited code.


## 2. Why a VAE changes the autoencoder

A regular autoencoder can reconstruct real inputs well because each latent code came from a real image. The decoder practices on codes produced by the encoder during training.

Generation asks for something harder:

```text
random latent code z → decoder → new image?
```

A regular autoencoder is not forced to make every random point in latent space meaningful. The encoder might place real image codes in separated islands. The empty regions between those islands are **latent gaps**: places where the decoder did not learn a reliable meaning.

If a random latent point lands in one of those gaps, it may correspond to a hidden representation the decoder has not learned from. The output can be blurry, broken, or not digit-like, even if reconstructions of real images look good.

A VAE changes the middle of the model so the encoder predicts a distribution instead of one fixed code.

```text
regular autoencoder:
x → fixed latent code z

variational autoencoder:
x → latent distribution q(z|x) → sampled latent code z
```

Formula view:

$$
q_\phi(z \mid x) \quad \text{instead of one fixed } z
$$

The VAE will use a simple **prior** distribution as the target shape for latent space:

$$
p(z) = \mathcal{N}(0, I)
$$

Read this as: before looking at any image, we want random latent codes $z$ to come from a standard normal cloud centered at 0 with unit spread. This matters because useful generation needs random `z` values to land in regions the decoder understands. That pressure makes the latent space more organized and sampleable.


## 3. Probability basics for VAE latent clouds

A VAE uses probability language because the encoder does not choose one exact hidden code immediately. For each input image, it describes a small cloud of likely latent codes, then samples one point from that cloud.

### Distribution vs sample

A **probability distribution** is the whole pattern of possible values and how likely each value is. A **sample** is one value picked from that distribution.

Small everyday example:

| possible outcome | probability |
| --- | ---: |
| sunny | 60% |
| cloudy | 30% |
| rainy | 10% |

The distribution is the whole table. Tomorrow's actual weather is one sample from that table.

For a VAE, the possible values are numeric latent codes $z$, not weather words. Picture the distribution as a cloud:

```text
center of cloud = likely z values
edges of cloud  = less likely z values
sample z        = one picked point from the cloud
```

### Normal clouds

A one-dimensional normal cloud can be written as:

$$
z \sim \mathcal{N}(\mu, \sigma^2)
$$

Read this as: sample $z$ from a normal distribution with center $\mu$ and variance $\sigma^2$.

Key terms:

- $\mu$ or `mu` = the mean, or center of the cloud.
- $\sigma$ or `std` = the standard deviation, or usable spread of the cloud.
- $\sigma^2$ = the variance, another way to describe spread.
- $\mathcal{N}(0, I)$ = the standard normal cloud used as the VAE prior: centered at 0 with unit spread in each latent dimension.

### Reading `q_phi(z | x)`

The encoder distribution is written:

$$
q_\phi(z \mid x)
$$

Use this map:

```text
q_phi( z | x )
  |    |   |
  |    |   already known input image
  |    possible latent code being scored or sampled
  encoder-side probability object with parameters phi
```

So $q_\phi(z \mid x)$ means:

```text
for this image x, the encoder's cloud of possible latent codes z
```

The thing before the vertical bar is what the distribution is **over**. Here, it is over possible $z$ values. The thing after the bar is what we already know. Here, the distribution depends on image $x$.

The sampling notation makes this clearer:

$$
z \sim q_\phi(z \mid x)
$$

Read it as:

```text
sample one z from the encoder's latent cloud for image x
```

### One cloud per image, one shared encoder, one global prior

A common confusion is whether there is one latent cloud for the whole dataset, one cloud per digit type, or one cloud per image. In this VAE, the most useful answer is:

```text
one trained encoder network
many image-specific latent clouds
one global prior cloud
```

For each input image $x_i$, the same encoder network outputs that image's own `mu` and `logvar`:

```text
x_i -> encoder -> mu_i, logvar_i -> q_phi(z | x_i)
```

So a batch of 128 images gives 128 image-specific latent clouds. If `latent_dim = 20`, then `mu` and `logvar` each have shape `[128, 20]`: one 20-number center and one 20-number spread description per image.

This does **not** mean the model explicitly creates one cloud per digit class such as `0`, `1`, or `7`. The notebook's VAE is not given digit labels. It only sees pixels. After training, similar-looking digits may end up near each other in latent space, but that is an emergent pattern, not a manually assigned class cloud.

The prior $p(z) = \mathcal{N}(0, I)$ is different. It is one global default cloud used for random generation:

```text
sample z from p(z) -> decoder -> new image
```

So keep these three levels separate:

| Level | Meaning | Example |
| --- | --- | --- |
| image-specific cloud | likely latent codes for one image | $q_\phi(z \mid x_i)$ |
| shared encoder | one network that creates those clouds for all images | `model.encode(x)` |
| global prior | simple default cloud used for generation | $p(z)=\mathcal{N}(0,I)$ |

Good enough for now: $q_\phi(z \mid x)$ is per image; $p(z)$ is global. KL loss keeps the per-image clouds close enough to the global prior that random samples are likely to land somewhere the decoder understands.

### Why does the prior need to match the clouds?

The decoder can be used in two different ways. During reconstruction, the notebook calls the whole model on a real image:

```text
real image x -> encoder -> q_phi(z | x) -> sample z -> decoder -> reconstruction x_hat
```

During generation, the practical notebook can skip the encoder and call the decoder directly:

```python
sample = torch.randn(64, latent_dim, device=device)
sample = model.decode(sample).cpu()
```

That code starts from random latent codes, not from real images. `torch.randn(...)` samples from the same standard-normal shape as the prior $p(z)=\mathcal{N}(0,I)$.

So the VAE wants the training-time latent clouds and the generation-time random samples to overlap:

```text
q_phi(z | x) clouds = where the decoder practices during reconstruction
p(z) samples        = where we sample from during generation
KL loss             = pressure to make those regions overlap
```

This is not a hard guarantee that every random $z$ will produce a perfect digit. It is a training pressure. If training works well, random values sampled from $\mathcal{N}(0,I)$ are likely to land in regions the decoder has learned to use.

The VAE cannot make every image cloud exactly equal to the prior either. If every $q_\phi(z \mid x)$ became identical to $p(z)$, then $z$ would carry almost no image-specific information and reconstructions would be poor. The useful compromise is:

```text
different enough to reconstruct each image
close enough to p(z) for random sampling to work
```

### Notation checkpoint

| Symbol | How to read it | Plain meaning | Code connection |
| --- | --- | --- | --- |
| $x$ | x | the original input image | `images` or `x` |
| $\hat{x}$ | x-hat | the reconstructed version of the input | `recon_x` or `recon_batch` |
| $z$ | z | one sampled latent code | `z` |
| $\mu$ | mu | center of the latent cloud | `mu` from `fc21` |
| $\sigma$ | sigma | standard deviation, or usable spread | `std` |
| $\log \sigma^2$ | log variance | stored spread output before conversion to `std` | `logvar` from `fc22` |
| $q_\phi(z \mid x)$ | q of z given x | encoder cloud for this input image | described by `mu` and `logvar` |
| $p(z)$ | p of z | default prior cloud for generation | usually `N(0, I)` or `torch.randn(...)` |
| $p_\theta(x \mid z)$ | p of x given z | decoder's probability model for images from this latent code | decoder output `recon_x` |

One common notation annoyance: $q_\phi(z \mid x)$ can mean the whole cloud, or the density value at one particular $z$, depending on context. For now, read it as the whole encoder cloud unless the text is clearly evaluating a score.


## 4. VAE encoder output: `mu` and `logvar`

In the practical notebook, the encoder has two output heads:

```python
self.fc21 = nn.Linear(400, latent_dim)
self.fc22 = nn.Linear(400, latent_dim)
```

The formula idea is:

$$
q_\phi(z \mid x) = \mathcal{N}(\mu, \sigma^2)
$$

Read this as: the encoder describes a normal latent cloud for this image, using a center $\mu$ and spread $\sigma$. In the full 20-dimensional VAE, this happens once per latent coordinate.

The first head outputs `mu`, the center of the latent cloud. The second head outputs ordinary neural-network numbers that the code names `logvar`.

The key idea is:

> `fc22` does not output variance directly. It outputs unconstrained numbers that we treat as log-variance.

Formula view:

$$
\text{logvar} = \log \sigma^2
$$

$$
\sigma^2 = e^{\text{logvar}}
$$

$$
\sigma = e^{0.5 \cdot \text{logvar}}
$$

This is useful because a neural network can output any real number: negative, zero, or positive. But variance cannot be negative, because a negative spread does not make sense. By predicting `logvar`, the model can output any real number first, and the code can convert it into a positive spread later.

The practical notebook does that conversion here:

```python
std = torch.exp(0.5 * logvar)
```

So `mu` moves the cloud, and `logvar` becomes the cloud's width after `exp(...)`. Training teaches both outputs to become useful for reconstruction and sampling.


In [2]:
# Tiny numeric example: convert logvar into variance and standard deviation.

import math

for logvar in [-2.0, 0.0, 2.0]:
    variance = math.exp(logvar)
    std = math.exp(0.5 * logvar)
    print(f"logvar={logvar:>4.1f}  variance={variance:>5.2f}  std={std:>5.2f}")


logvar=-2.0  variance= 0.14  std= 0.37
logvar= 0.0  variance= 1.00  std= 1.00
logvar= 2.0  variance= 7.39  std= 2.72


## 5. Reparameterization trick

After the encoder predicts `mu` and `logvar`, the VAE needs to sample one latent code `z` from the latent cloud. The practical notebook does that with this code:

```python
std = torch.exp(0.5 * logvar)
eps = torch.randn_like(std)
z = mu + eps * std
```

Formula view:

$$
\varepsilon \sim \mathcal{N}(0, I)
$$

$$
z = \mu + \sigma \odot \varepsilon
$$

Read it as:

```text
sampled latent code = learned center + learned spread × random noise
```

The symbol $\odot$ means element-by-element multiplication. In code, that is the `eps * std` part. It is not the same as Python's `@` operator. `*` or $\odot$ keeps one value per latent coordinate; `@` does dot-product or matrix multiplication and combines coordinates together.

Tiny operator comparison:

```python
std = torch.tensor([2, 3, 4])
eps = torch.tensor([10, 20, 30])

std * eps  # tensor([20, 60, 120])  element-by-element, same shape
std @ eps  # tensor(200)            dot product, coordinates combined
```

For reparameterization, we need elementwise multiplication because each latent coordinate gets its own random noise scaled by its own learned spread.

Each part has a job:

- `eps` or $\varepsilon$ is random standard-normal noise. It is not learned.
- `mu` or $\mu$ is the center predicted by the encoder.
- `logvar` is the spread information predicted by the encoder.
- `std` or $\sigma$ is computed from `logvar` and gives the usable spread.
- `z` is the sampled latent code that goes into the decoder.

The trick is that the randomness is isolated in `eps`. Once `eps` has been sampled, `z = mu + eps * std` is ordinary tensor math, so the loss can still send learning signals back through `mu` and `logvar`.


### Is `z` a point, a vector, or a matrix?

For one image, `z` is one sampled point in latent space. Because this VAE uses `latent_dim = 20`, that point needs 20 coordinates, so PyTorch stores it as a 20-number vector.

```text
one image: z shape = [latent_dim] = [20]
```

During training, the model processes a batch of images at once. Then PyTorch stacks one sampled point per image into a matrix-like tensor.

```text
batch of images: z shape = [batch_size, latent_dim]
example from the practical notebook: z shape = [128, 20]
```

So both statements are true: `z` is a sampled point in the latent cloud for one image, and a batch of `z` values is stored as a 2D tensor with one row per image.


In [3]:
# Tiny shape example: three sampled latent points in a 2D latent space.

import torch

torch.manual_seed(1)

mu = torch.tensor([
    [0.0, 0.0],
    [5.0, 5.0],
    [-2.0, 1.0],
])
logvar = torch.zeros_like(mu)
std = torch.exp(0.5 * logvar)
eps = torch.randn_like(std)
z = mu + eps * std

print(f"mu shape:  {tuple(mu.shape)}")
print(f"eps shape: {tuple(eps.shape)}")
print(f"z shape:   {tuple(z.shape)}")
print(z)


mu shape:  (3, 2)
eps shape: (3, 2)
z shape:   (3, 2)
tensor([[ 0.6614,  0.2669],
        [ 5.0617,  5.6213],
        [-2.4519,  0.8339]])


## 6. Reconstruction loss

After the decoder receives `z`, it produces a reconstruction `x_hat`. Reconstruction loss measures how different `x_hat` is from the original input image `x`.

```text
z → decoder → x_hat
compare x_hat with x
```

Probability notation writes the decoder as:

$$
p_\theta(x \mid z)
$$

Read this as: given latent code $z$, how likely are different output images $x$ under the decoder?

The reconstruction-loss idea is the negative log-likelihood of the real image pixels:

$$
\mathcal{L}_{\text{recon}} = -\log p_\theta(x \mid z)
$$

Plain English: make the actual input image less surprising under the decoder.

In the practical notebook, this is the reconstruction term:

```python
reconstruction_loss = F.binary_cross_entropy(
    recon_x,
    x.view(-1, 784),
    reduction="sum",
)
```

For MNIST-style pixels in `[0, 1]`, BCE can be written as:

$$
\mathcal{L}_{\text{BCE}}
= -\sum_i \left[x_i \log(\hat{x}_i) + (1 - x_i)\log(1 - \hat{x}_i)\right]
$$

You do not need to memorize this formula. Read it as: compare each rebuilt pixel $\hat{x}_i$ with the matching original pixel $x_i$, penalize confident wrong pixel predictions, then add the pixel penalties.

This is not classifier loss. The target is not a digit label like `7`; the target is the original input image itself.


## 7. KL divergence intuition

Reconstruction loss is only half of the VAE objective. If the model only cared about reconstruction, each image cloud could move wherever it wants. That can make reconstructions good, but random generation unreliable.

The KL term compares:

$$
q_\phi(z \mid x) \quad \text{and} \quad p(z)
$$

```text
q_phi(z | x) = the encoder's latent cloud for this input image
p(z)         = the simple prior cloud we sample from later, usually N(0, I)
```

For this notebook, read KL divergence as:

```text
KL loss = penalty for this image's latent cloud being too different from the prior cloud
```

### The two checks KL is doing

For one latent coordinate, the prior cloud is:

$$
p(z) = \mathcal{N}(0, 1)
$$

That means the target shape is:

```text
center = 0
variance = 1
```

The encoder cloud for one image is:

$$
q_\phi(z \mid x) = \mathcal{N}(\mu, \sigma^2)
$$

That means:

```text
center = mu
variance = sigma^2
```

So the KL term mainly asks two questions:

```text
1. Is the center mu too far from 0?
2. Is the variance too different from 1?
```

### Center penalty: `mu.pow(2)`

If `mu = 0`, the cloud is centered where the prior is centered. If `mu = 5` or `mu = -5`, the cloud is far away from the prior center.

The formula uses $\mu^2$ because squaring makes distance from zero positive:

```text
mu =  0 -> mu^2 = 0
mu =  5 -> mu^2 = 25
mu = -5 -> mu^2 = 25
```

In code:

```python
mu.pow(2)
```

means:

```text
penalize cloud centers that move far from 0
```

### Spread penalty: `logvar.exp()` and `-logvar`

The prior wants variance `1`. The code stores spread as `logvar`, where:

$$
\text{logvar} = \log \sigma^2
$$

So this code converts `logvar` back into variance:

```python
logvar.exp()
```

Examples:

```text
logvar = 0  -> variance = exp(0)  = 1.00   good
logvar = 2  -> variance = exp(2)  = 7.39   too wide
logvar = -2 -> variance = exp(-2) = 0.14   too narrow
```

Both extremes are a problem:

```text
too wide   -> z is very noisy, so reconstruction gets unstable
too narrow -> the cloud becomes almost a fixed code, so latent space can scatter again
```

So the spread part says:

```text
keep variance close to 1
```

### Friendly formula first

For one latent coordinate, write the KL term as:

$$
\text{KL} = \frac{1}{2}\left(\mu^2 + e^{\text{logvar}} - 1 - \text{logvar}\right)
$$

Read it as:

```text
KL = center penalty + spread penalty, adjusted so the perfect match gives 0
```

The perfect match is:

```text
mu = 0
logvar = 0
variance = exp(0) = 1
```

Plugging that in:

```text
KL = 0.5 * (0^2 + exp(0) - 1 - 0)
KL = 0.5 * (0 + 1 - 1 - 0)
KL = 0
```

### Why the PyTorch code looks rearranged

The practical notebook computes the same expression in a rearranged form:

```python
kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
```

This looks different, but it is the same math. Distribute the `-0.5`:

```text
-0.5 * (1 + logvar - mu^2 - exp(logvar))
= 0.5 * (mu^2 + exp(logvar) - 1 - logvar)
```

So these match:

```text
friendly formula: 0.5 * (mu^2 + exp(logvar) - 1 - logvar)
PyTorch code:     -0.5 * (1 + logvar - mu^2 - exp(logvar))
```

### Tiny numeric examples

```text
Perfect prior-like cloud:
mu = 0, logvar = 0
KL = 0
```

```text
Center too far:
mu = 3, logvar = 0
KL = 0.5 * (9 + 1 - 1 - 0)
KL = 4.5
```

```text
Spread too wide:
mu = 0, logvar = 2
KL = 0.5 * (0 + 7.39 - 1 - 2)
KL = 2.195
```

### Part-by-part code map

| Code part | Plain meaning |
| --- | --- |
| `mu.pow(2)` | penalizes centers far from `0` |
| `logvar.exp()` | converts `logvar` back into variance and helps penalize overly wide clouds |
| `-logvar` in the friendly formula | helps penalize overly narrow clouds |
| `-1` in the friendly formula | makes the penalty become `0` at the ideal match |
| `torch.sum(...)` | adds the penalty across latent dimensions and batch items |

The full VAE tradeoff is:

```text
total loss = reconstruction loss + KL loss

reconstruction loss: rebuild x well
KL loss: keep latent clouds close enough to p(z) for random sampling
```

Good enough for now:

```text
KL loss penalizes mu far from 0 and variance far from 1.
That keeps each image-specific cloud near the prior.
This makes random z from torch.randn(...) more likely to decode into something meaningful.
```


In [4]:
# Tiny numeric example: KL grows when mu moves from 0 or variance moves from 1.

import math


def kl_to_standard_normal(mu: float, variance: float) -> float:
    """Compute KL(N(mu, variance) || N(0, 1)) for one latent dimension.

    Args:
        mu: Center of the one-dimensional latent cloud.
        variance: Spread-squared of the latent cloud. Must be positive.

    Returns:
        KL penalty against the standard normal distribution.
    """
    return 0.5 * (mu**2 + variance - 1.0 - math.log(variance))


examples = [
    (0.0, 1.0),
    (2.0, 1.0),
    (0.0, 0.25),
    (0.0, 4.0),
    (2.0, 4.0),
]

for mu, variance in examples:
    kl = kl_to_standard_normal(mu=mu, variance=variance)
    print(f"mu={mu:>3.1f}  variance={variance:>4.2f}  KL={kl:>5.2f}")


mu=0.0  variance=1.00  KL= 0.00
mu=2.0  variance=1.00  KL= 2.00
mu=0.0  variance=0.25  KL= 0.32
mu=0.0  variance=4.00  KL= 0.81
mu=2.0  variance=4.00  KL= 2.81


## 8. Likelihood in plain language

VAE explanations often say: "maximize the likelihood of the real image." That sentence can sound abstract, so translate it as a practical question:

```text
Could this VAE naturally generate or reconstruct an image like this real input x?
```

The decoder likelihood is written as:

$$
p_\theta(x \mid z)
$$

Read this as: if the decoder receives latent code $z$, how likely is image $x$?

If the VAE usually produces random gray blobs, then a clear MNIST digit has low likelihood under the model. The real digit is surprising for that model because it does not look like what the model tends to produce.

If the VAE produces digit-like images with similar strokes and pixels, then a real MNIST digit has higher likelihood under the model. The image is less surprising because it fits what the model has learned to produce.

Important: this is not class-label recognition. "High likelihood" does not mean the model says the digit class is `7`. It means the model gives high probability to the actual image pixels.

For a VAE, the generation story is:

```text
sample z → decoder → possible image x
```

So "how likely is image `x`?" means: if we sampled latent codes `z` and decoded them, would this model often produce something close to `x`?


## 9. Optional later theory: evidence, integrals, and ELBO

You can understand and run the VAE implementation without this section. The core notebook path is already enough for now:

```text
image x -> encoder -> mu/logvar -> sample z -> decoder -> reconstruction
total loss = reconstruction loss + KL loss
generation = random z from p(z) -> decoder
```

The deeper theory asks a different question:

```text
How likely is image x under the whole generative model, after considering all possible latent causes z?
```

Some texts call that full likelihood the **evidence**, written $p_	heta(x)$. Computing it exactly would require adding contributions from all possible latent codes $z$. In continuous latent space, that "add over all z" idea is written as an integral:

$$
p_	heta(x) = \int p_	heta(x \mid z)\,p(z)\,dz
$$

For now, read this only as:

```text
p_theta(x) = total support for image x after considering all possible z values
```

That exact computation is too expensive for this neural-network training loop. The ELBO is the practical objective VAEs use instead. In code, the notebook does not compute the integral directly; it trains with the two terms you have already studied:

```text
negative ELBO ≈ reconstruction loss + KL loss
```

Good enough for now: treat integrals and formal ELBO derivations as later theory. The implementation-level mental model is reconstruction plus KL-regularized sampling.


## 10. ELBO intuition (still optional)

Because direct $\log p_\theta(x)$ is hard, VAEs optimize a related objective called the **ELBO**, short for **Evidence Lower BOund**.

Beginner translation:

```text
ELBO = a trainable score for how well the VAE explains x
```

It is called a lower bound because it stays below the true $\log p_\theta(x)$, but improving the ELBO usually improves the model. The VAE can compute and optimize this lower-bound score even when the exact evidence is too hard.

Simple formula view:

$$
\log p_\theta(x)
\ge
\mathbb{E}_{q_\phi(z \mid x)}[\log p_\theta(x \mid z)]
-
D_{KL}\left(q_\phi(z \mid x)\,\|\,p(z)\right)
$$

Read it as:

```text
true image likelihood is at least:
reconstruction reward minus KL penalty
```

Training code usually minimizes losses instead of maximizing rewards, so we use **negative ELBO**:

$$
-\text{ELBO} = \mathcal{L}_{\text{recon}} + \mathcal{L}_{\text{KL}}
$$

That is the reason the practical notebook returns the sum of the two terms:

```python
return reconstruction_loss + kl_loss
```

High-level meaning:

- Reconstruction loss asks: can the decoder rebuild this image from the sampled latent code?
- KL loss asks: did the encoder keep the image's latent cloud near the simple prior cloud?
- Negative ELBO combines both goals into one trainable loss.


## 11. Mapping the math back to `loss_function()`

The practical notebook's loss function is a compact implementation of negative ELBO.

```python
reconstruction_loss = F.binary_cross_entropy(
    recon_x,
    x.view(-1, 784),
    reduction="sum",
)

kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

return reconstruction_loss + kl_loss
```

Line-by-line map:

| Code | Math idea | Plain meaning |
| --- | --- | --- |
| `recon_x` | $\hat{x}$ | decoder's rebuilt pixels |
| `x.view(-1, 784)` | $x$ | original image pixels flattened to match `recon_x` |
| `F.binary_cross_entropy(...)` | $\mathcal{L}_{\text{recon}}$ | pixel reconstruction penalty |
| `mu` | $\mu$ | center of the encoder's latent cloud |
| `logvar` | $\log \sigma^2$ | stored spread value for the latent cloud |
| `logvar.exp()` | $\sigma^2$ | variance recovered from log variance |
| `mu.pow(2)` | $\mu^2$ | penalty for moving the center away from 0 |
| `kl_loss` | $\mathcal{L}_{\text{KL}}$ | penalty for drifting away from $\mathcal{N}(0, I)$ |
| `return reconstruction_loss + kl_loss` | $-\text{ELBO}$ | rebuild well while keeping latent space sampleable |

Compact mental model:

$$
\boxed{\text{VAE loss} = \text{rebuild the image} + \text{organize the latent space}}
$$
